# V2 Complex Attributes

## 목적

V1 회귀는 처음 보는 단지에서 성능이 크게 떨어졌다(MAPE 약 18%, 본 단지 약 9%).
이름형 범주 Feature가 위치 ID처럼 쓰였기 때문이다. 이 노트북은 **단지 자체를 설명하는 속성**을 만들어
처음 보는 단지에도 일반화되는 Feature를 준비한다.

| 데이터 | 출처 | 단위 | 시점 처리 |
|---|---|---|---|
| 건축물대장 표제부·총괄표제부 | 국토부 건축HUB (`04e_collect_building_register.py`, 2026-10-22 수집) | 동·단지 | 물리적 속성이라 고정값 (리모델링 외 변하지 않음) |
| 공동주택 공시가격 2020~2026 | data.go.kr 3073746 (`05b_download_official_price.py`) | 호 | 포털 파일은 6월 말 조정공시 반영본 → **Y년 값은 Y년 7월 1일부터** |
| 서울시 정비사업 추진현황 (2026-06, 2026-03) | 서울 열린데이터 OA-22856 | 구역 | 단계별 고시일이 계약일 이전인 단계만 |
| 지하철역 → 업무지구 통근시간 | ODsay (`05b_collect_commute.py`) | 역 | **모델에서 제외** — 웹에서 단지별 실시간 조회로 제공 (정리 참고) |

### 원칙

- 거래 단지와 외부 자료는 **필지 키(시군구 5 + 법정동 5 + 본번 4 + 부번 4)**로 연결한다.
- 실제 거래금액은 Feature 계산에 쓰지 않는다. 5장 점검에서만 비교용으로 읽는다.

In [1]:
# ============================================================
# 경로·데이터
# ============================================================

from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", message="The 'generic' unit for NumPy timedelta")
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")


def find_project_root(start=Path.cwd()):
    for path in [start, *start.parents]:
        if (path / "data").exists():
            return path
    raise FileNotFoundError("프로젝트 루트에서 data 폴더를 찾을 수 없습니다.")


PROJECT_DIR = find_project_root()
EXTERNAL_DIR = PROJECT_DIR / "data" / "external"
PROCESSED_DIR = PROJECT_DIR / "data" / "v2_5year_seoul" / "processed"

trades = pd.read_parquet(
    PROCESSED_DIR / "df_fe_base.parquet",
    columns=["transaction_id", "계약일자", "단지일련번호", "단지명", "시군구명", "법정동", "지번",
             "법정동시군구코드", "법정동읍면동코드", "법정동본번코드", "법정동부번코드", "전용면적", "건축년도"],
)
trades["lot"] = (
    trades["법정동시군구코드"].astype(str).str.zfill(5) + trades["법정동읍면동코드"].astype(str).str.zfill(5)
    + trades["법정동본번코드"].astype(str).str.zfill(4) + trades["법정동부번코드"].astype(str).str.zfill(4)
)
complexes = trades.drop_duplicates("단지일련번호")[["단지일련번호", "단지명", "시군구명", "법정동", "지번", "lot", "건축년도"]]

print("거래:", trades.shape, "| 단지:", len(complexes))

거래: (234302, 14) | 단지: 7871


## 1. 단지 물리 속성 (건축물대장)

In [2]:
# ============================================================
# 표제부(동별) → 필지 단위 집계, 총괄표제부(단지 전체)에서 주차·용적률 보강
# ============================================================

def lot_key(d):
    return d["sigunguCd"] + d["bjdongCd"] + d["bun"].str.zfill(4) + d["ji"].str.zfill(4)


def num(d, cols):
    for c in cols:
        d[c] = pd.to_numeric(d[c], errors="coerce").fillna(0)
    return d


PARK = ["indrAutoUtcnt", "oudrAutoUtcnt", "indrMechUtcnt", "oudrMechUtcnt"]

title = pd.read_parquet(EXTERNAL_DIR / "building_register_apt_seoul_20261022.parquet")
title = num(title, ["hhldCnt", "grndFlrCnt", "ugrndFlrCnt", "rideUseElvtCnt", "emgenUseElvtCnt", "vlRat", "bcRat"] + PARK)
title["lot"] = lot_key(title)
title["사용승인연도"] = pd.to_numeric(title["useAprDay"].astype(str).str[:4], errors="coerce")
title["주차"] = title[PARK].sum(axis=1)
res = title[title["hhldCnt"] > 0]

by_lot = res.groupby("lot").agg(
    동수=("hhldCnt", "size"),
    대장세대수=("hhldCnt", "sum"),
    최고층=("grndFlrCnt", "max"),
    평균층=("grndFlrCnt", "mean"),
    지하층=("ugrndFlrCnt", "max"),
    승강기=("rideUseElvtCnt", "sum"),
    표제_용적률=("vlRat", lambda s: s[s > 0].max() if (s > 0).any() else np.nan),
    표제_건폐율=("bcRat", lambda s: s[s > 0].max() if (s > 0).any() else np.nan),
    표제_주차=("주차", "sum"),
    사용승인연도=("사용승인연도", "min"),
)

recap = pd.read_parquet(EXTERNAL_DIR / "building_register_recap_seoul_20261022.parquet")
recap = num(recap, ["hhldCnt", "totPkngCnt", "vlRat", "bcRat", "mainBldCnt"] + PARK)
recap["lot"] = lot_key(recap)
recap["주차"] = recap["totPkngCnt"].where(recap["totPkngCnt"] > 0, recap[PARK].sum(axis=1))
by_recap = recap.groupby("lot").agg(
    총괄_주차=("주차", "max"), 총괄_세대수=("hhldCnt", "max"),
    총괄_용적률=("vlRat", "max"), 총괄_건폐율=("bcRat", "max"),
)

attrs = complexes.merge(by_lot, left_on="lot", right_index=True, how="left") \
                 .merge(by_recap, left_on="lot", right_index=True, how="left")

# 단지 대표값: 총괄표제부 우선, 없으면 동별 표제부
attrs["세대수"] = attrs["총괄_세대수"].where(attrs["총괄_세대수"] > 0, attrs["대장세대수"])
attrs["주차대수"] = attrs["총괄_주차"].where(attrs["총괄_주차"] > 0, attrs["표제_주차"])
attrs["세대당_주차"] = (attrs["주차대수"] / attrs["세대수"]).where(attrs["주차대수"] > 0)
attrs["용적률"] = attrs["총괄_용적률"].where(attrs["총괄_용적률"] > 0, attrs["표제_용적률"])
attrs["건폐율"] = attrs["총괄_건폐율"].where(attrs["총괄_건폐율"] > 0, attrs["표제_건폐율"])
attrs["승강기_100세대"] = (attrs["승강기"] / attrs["대장세대수"] * 100).where(attrs["승강기"] > 0)

# 비현실적인 값은 결측 처리 (입력 오류)
attrs.loc[~attrs["세대당_주차"].between(0.05, 5), "세대당_주차"] = np.nan
attrs.loc[~attrs["용적률"].between(30, 1500), "용적률"] = np.nan
attrs.loc[~attrs["건폐율"].between(5, 100), "건폐율"] = np.nan
attrs.loc[attrs["최고층"] <= 0, ["최고층", "평균층"]] = np.nan

ATTR_FEATURES = ["동수", "세대수", "최고층", "평균층", "지하층", "승강기_100세대", "세대당_주차", "용적률", "건폐율"]
print("대장 연결 단지:", attrs["동수"].notna().sum(), "/", len(attrs))
display(attrs[ATTR_FEATURES].describe(percentiles=[0.05, 0.5, 0.95]).T.round(2))
display(attrs[ATTR_FEATURES].notna().mean().round(3).to_frame("결측 아님 비율"))

대장 연결 단지: 7780 / 7871


,count,mean,std,min,5%,50%,95%,max
동수,7780.0,3.22,6.13,1.00,1.00,1.00,13.00,122.0
세대수,7782.0,238.58,521.89,2.00,10.00,56.00,1027.65,12032.0
최고층,7779.0,12.85,7.11,2.00,5.00,12.00,25.00,69.0
평균층,7779.0,12.14,6.40,2.00,5.00,11.00,23.00,69.0
지하층,7780.0,1.30,1.26,0.00,0.00,1.00,4.00,9.0
승강기_100세대,5518.0,5.19,3.35,0.04,0.81,5.00,11.11,34.0
세대당_주차,6910.0,1.25,0.61,0.05,0.56,1.12,2.53,5.0
용적률,7129.0,311.45,144.34,31.60,192.24,268.93,629.54,1472.0
건폐율,7132.0,40.75,15.48,5.85,18.81,41.14,59.98,100.0


,결측 아님 비율
동수,0.988
세대수,0.989
최고층,0.988
평균층,0.988
지하층,0.988
승강기_100세대,0.701
세대당_주차,0.878
용적률,0.906
건폐율,0.906


In [3]:
# ============================================================
# 연결 검증: 대장 최초 사용승인연도 vs 거래 데이터 건축년도
# - 같은 필지에 신축과 구축이 섞인 경우(재건축 후 신축 등)를 제외하면 대부분 같아야 한다.
# ============================================================

chk = attrs.dropna(subset=["사용승인연도"])
gap = (chk["사용승인연도"] - pd.to_numeric(chk["건축년도"], errors="coerce")).abs()
print(f"사용승인연도 = 건축년도 (±1년): {(gap <= 1).mean():.1%}  (n={len(chk)})")
display(chk.loc[gap > 1, ["단지명", "법정동", "지번", "건축년도", "사용승인연도", "세대수"]].head(8))

사용승인연도 = 건축년도 (±1년): 99.8%  (n=7766)


,단지명,법정동,지번,건축년도,사용승인연도,세대수
20077,극동2,광장동,218-1,1989,1985.0,1344.0
59931,신동아아파트1,방학동,271-1,1986,1988.0,604.0
60959,쌍문동LG트윈빌1,쌍문동,19,1999,1984.0,20.0
95537,홍연,연희동,708-4,1981,1984.0,15.0
124436,방화동동부센트레빌(875번지),방화동,875,2003,1997.0,407.0
124604,방화그린,방화동,839,1993,1990.0,2607.0
137595,대림2,신도림동,642,2001,1999.0,2304.0
139781,한성,고척동,227-1,1986,1989.0,80.0


## 2. 공시가격 (2020~2026, 서울 아파트)

- 단지 × 공시연도별 **㎡당 공시가격 중위값**과, 면적대(60/85/135㎡) 별 중위값을 만든다.
- 거래일 d에는 **d 시점에 공개돼 있던 가장 최근 공시연도**(Y년 값 → Y년 7월 1일부터)를 붙인다.

In [4]:
# ============================================================
# 공시가격 집계 (단지·면적대·연도)
# ============================================================

AREA_BINS, AREA_LABELS = [0, 60, 85, 135, np.inf], ["≤60", "60~85", "85~135", ">135"]

frames = []
for path in sorted((EXTERNAL_DIR / "official_price").glob("apt_official_price_seoul_*.parquet")):
    g = pd.read_parquet(path, columns=["기준연도", "법정동코드", "본번", "부번", "전용면적", "공시가격"])
    g["lot"] = (g["법정동코드"].astype(str).str.zfill(10) + g["본번"].astype(str).str.zfill(4)
                + g["부번"].astype(str).str.zfill(4))
    g["m2당"] = pd.to_numeric(g["공시가격"], errors="coerce") / pd.to_numeric(g["전용면적"], errors="coerce") / 10_000  # 만원/㎡
    g["면적대"] = pd.cut(pd.to_numeric(g["전용면적"], errors="coerce"), AREA_BINS, labels=AREA_LABELS)
    frames.append(g[["기준연도", "lot", "면적대", "m2당"]])
gongsi = pd.concat(frames, ignore_index=True).dropna(subset=["m2당"])
gongsi["기준연도"] = gongsi["기준연도"].astype(int)

price_lot = gongsi.groupby(["lot", "기준연도"])["m2당"].agg(공시_m2당_단지="median", 공시_호수="size").reset_index()
price_band = gongsi.groupby(["lot", "기준연도", "면적대"], observed=True)["m2당"].median().rename("공시_m2당_면적대").reset_index()

print("공시 호 수:", len(gongsi), "| 연도:", sorted(gongsi["기준연도"].unique()))
display(price_lot.groupby("기준연도")["공시_m2당_단지"].describe(percentiles=[0.5]).round(1))

공시 호 수: 12291587 | 연도: [np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]


,count,mean,std,min,50%,max
기준연도,,,,,,
2020,9223.0,532.0,361.9,152.6,438.0,3192.1
2021,9364.0,621.3,399.8,120.4,523.1,2978.5
2022,9514.0,704.4,471.9,125.0,588.8,3687.2
2023,9532.0,597.4,341.5,118.2,518.5,3022.8
2024,9637.0,608.3,356.3,116.3,524.3,3293.8
2025,9709.0,642.3,404.0,118.1,541.9,4079.3
2026,9772.0,726.2,535.0,127.2,578.2,6793.2


In [5]:
# ============================================================
# 거래일 기준 공개된 공시연도 붙이기
# ============================================================

RELEASE_MONTH = 7  # Y년 공시(조정 반영본) → Y년 7월 1일부터


def available_year(dates):
    return np.where(dates.dt.month >= RELEASE_MONTH, dates.dt.year, dates.dt.year - 1)


def gongsi_features(tr, price_lot, price_band):
    t = tr[["transaction_id", "lot", "계약일자", "전용면적"]].copy()
    t["공시연도"] = available_year(t["계약일자"])
    t["면적대"] = pd.cut(t["전용면적"].astype(float), AREA_BINS, labels=AREA_LABELS)

    t = t.merge(price_lot, left_on=["lot", "공시연도"], right_on=["lot", "기준연도"], how="left").drop(columns="기준연도")
    prev = price_lot.assign(기준연도=price_lot["기준연도"] + 1).rename(columns={"공시_m2당_단지": "공시_m2당_전년"})
    t = t.merge(prev[["lot", "기준연도", "공시_m2당_전년"]], left_on=["lot", "공시연도"],
                right_on=["lot", "기준연도"], how="left").drop(columns="기준연도")
    t = t.merge(price_band, left_on=["lot", "공시연도", "면적대"], right_on=["lot", "기준연도", "면적대"],
                how="left").drop(columns="기준연도")

    band = t["공시_m2당_면적대"].fillna(t["공시_m2당_단지"])
    out = pd.DataFrame({"transaction_id": t["transaction_id"].to_numpy()})
    out["공시연도"] = t["공시연도"].to_numpy()
    out["공시_m2당_단지"] = t["공시_m2당_단지"].to_numpy()
    out["공시_m2당_면적대"] = band.to_numpy()
    out["공시기반_추정가"] = (band * t["전용면적"].astype(float)).to_numpy()  # 만원
    out["공시_전년대비"] = ((t["공시_m2당_단지"] / t["공시_m2당_전년"] - 1) * 100).to_numpy()
    return out


gongsi_trade = gongsi_features(trades, price_lot, price_band)
print("공시가격이 붙은 거래:", f"{gongsi_trade['공시_m2당_단지'].notna().mean():.1%}")
display(gongsi_trade.groupby("공시연도")[["공시_m2당_단지", "공시_전년대비"]].agg(["count", "median"]).round(1))

공시가격이 붙은 거래: 97.1%


공시_m2당_단지        공시_전년대비       
         count median   count median
공시연도                                
2021     22682  683.2   22088   24.2
2022     20292  959.3   19235   13.9
2023     42592  755.3   41987  -20.6
2024     70597  785.0   69782    1.8
2025     71363  701.6   70675    4.7

## 3. 정비사업(재건축) 단계

서울시 정비사업 추진현황(2026-06 기준)의 재건축 구역을 거래 단지와 필지로 연결하고,
계약일 이전에 고시된 단계만으로 그 시점의 단계를 정한다.

| 단계 값 | 단계 |
|---|---|
| 0 | 정비구역 아님 (또는 구역지정 전) |
| 1 | 구역지정(정비계획) |
| 2 | 추진위원회 |
| 3 | 조합설립인가 |
| 4 | 사업시행인가 |
| 5 | 관리처분계획인가 |
| 6 | 이주 시작 |
| 7 | 착공 |

In [6]:
# ============================================================
# 정비구역 정리 + 필지 키 만들기
# - 지번 표기: '수송동146-12', '이촌동193-3,5'(여러 필지), '중곡4동18-24'(행정동), '양천구신정동1200'(구 이름 포함)
# ============================================================

COLS = ["CODE", "순번", "자치구", "구역명", "위치1", "위치2", "공공민간", "일반재촉", "사업유형", "사업추진단계",
        "기존가구수", "구역지정_최초", "구역지정_변경", "추진위원회", "조합설립", "건축심의", "사업시행_최초", "사업시행_변경",
        "관리처분_최초", "관리처분_변경", "이주시작", "이주종료", "착공", "건립세대_계", "분양", "임대"]


def read_jeongbi(path):
    d = pd.read_excel(path, header=None, skiprows=4)
    d.columns = COLS
    return d[d["CODE"].notna()].copy()


zones = read_jeongbi(EXTERNAL_DIR / "jeongbi" / "seoul_jeongbi_status_202606.xlsx")
zones_mar = read_jeongbi(EXTERNAL_DIR / "jeongbi" / "seoul_jeongbi_status_202603.xlsx")

STAGES = [("구역지정_최초", 1), ("추진위원회", 2), ("조합설립", 3), ("사업시행_최초", 4),
          ("관리처분_최초", 5), ("이주시작", 6), ("착공", 7)]
for col, _ in STAGES:
    zones[col] = pd.to_datetime(zones[col], errors="coerce")

rebuild = zones[zones["사업유형"].isin(["공동주택재건축", "아파트지구재건축"])].copy()


def parse_lot(text, gu):
    s = str(text).replace(" ", "").replace("번지", "")
    s = s.replace(gu, "")
    m = re.match(r"^(.+?[동가로])(\d+)(?:-(\d+))?", s)
    if not m:
        return None
    dong = re.sub(r"\d+동$", "동", m.group(1))     # 중곡4동 → 중곡동 (행정동 → 법정동 근사)
    dong = re.sub(r"본동$", "동", dong)             # 망우본동 → 망우동
    return dong, int(m.group(2)), int(m.group(3) or 0)


parsed = rebuild.apply(lambda r: parse_lot(r["위치1"], r["자치구"]), axis=1)
rebuild["법정동"] = parsed.map(lambda x: x[0] if x else None)
rebuild["본번"] = parsed.map(lambda x: x[1] if x else None)
rebuild["부번"] = parsed.map(lambda x: x[2] if x else None)

cx = complexes.assign(
    본번=trades.drop_duplicates("단지일련번호")["법정동본번코드"].astype(int).to_numpy(),
    부번=trades.drop_duplicates("단지일련번호")["법정동부번코드"].astype(int).to_numpy(),
)
link = rebuild.merge(cx[["단지일련번호", "시군구명", "법정동", "본번", "부번"]],
                     left_on=["자치구", "법정동", "본번", "부번"], right_on=["시군구명", "법정동", "본번", "부번"], how="left")

matched = link.dropna(subset=["단지일련번호"])
print(f"재건축 구역 {len(rebuild)}곳 중 거래 단지와 연결: {matched['CODE'].nunique()}곳 ({matched['단지일련번호'].nunique()}개 단지)")
display(link[link["단지일련번호"].isna()][["자치구", "구역명", "위치1", "사업추진단계"]].head(10))

재건축 구역 166곳 중 거래 단지와 연결: 132곳 (139개 단지)


,자치구,구역명,위치1,사업추진단계
9,성동구,장미아파트,성수동1가656-421,착공
13,광진구,신향빌라,중곡4동18-24,조합설립
20,중랑구,망우1(염광아파트),망우본동178-1,조합설립
21,중랑구,중화동우성타운,중화동195-2,건축심의
23,도봉구,방학신동아1단지아파트,방학동272,구역지정
29,구로구,길훈아파트,개봉동312-38,착공
30,서초구,반포주공1단지(3주구),반포동1109,착공
39,양천구,목동14단지,양천구신정동331,조합설립
51,강동구,삼익파크,길동54,관리처분
52,강서구,신안빌라,마곡동327-53,사업시행


In [7]:
# ============================================================
# 거래일 기준 단계 (고시일 < 계약일인 단계 중 가장 높은 단계)
# ============================================================

zone_of = matched[["단지일련번호", "CODE"] + [c for c, _ in STAGES]].drop_duplicates("단지일련번호")


def stage_features(tr, zone_of):
    t = tr[["transaction_id", "단지일련번호", "계약일자"]].merge(zone_of, on="단지일련번호", how="left")
    stage = np.zeros(len(t), dtype=int)
    for col, value in STAGES:
        stage = np.where(t[col].notna() & (t[col] < t["계약일자"]), np.maximum(stage, value), stage)
    out = pd.DataFrame({"transaction_id": t["transaction_id"].to_numpy(), "재건축_단계": stage})
    out["재건축_구역"] = (stage >= 1).astype(int)
    out["재건축_조합설립이후"] = (stage >= 3).astype(int)
    return out


stage_trade = stage_features(trades, zone_of)
display(pd.crosstab(trades["계약일자"].dt.year, stage_trade["재건축_단계"]))

# 3월 → 6월 사이 목록에서 사라진 구역 (준공·해제 등) — 과거 시점에는 있었지만 목록에 없는 구역의 규모 가늠
gone = set(zones_mar["CODE"]) - set(zones["CODE"])
print("2026-03 목록에는 있고 2026-06 목록에는 없는 구역:", len(gone))

재건축_단계,0,1,2,3,4,5,6
계약일자,,,,,,,
2021,15888,2,47,146,24,6,0
2022,11864,10,57,85,22,9,0
2023,33147,14,210,484,100,57,22
2024,53236,284,122,1065,313,107,47
2025,74270,469,553,1196,686,182,9
2026,38506,51,273,468,115,135,21


2026-03 목록에는 있고 2026-06 목록에는 없는 구역: 6


## 4. 누수 검증

- **공시가격:** 표본 계약일마다 그날 공개돼 있던 공시연도 자료만 남기고 다시 계산해 같은지 본다.
- **정비 단계:** 표본 계약일마다 그 이후 고시일을 지운 구역표로 다시 계산해 같은지 본다.
- **경계일 기대값:** 6/30 → 전년 공시, 7/1 → 당해 공시.

In [8]:
# ============================================================
# 재계산 비교 (계약일 200개 표본)
# ============================================================

rng = np.random.default_rng(3)
sample_dates = rng.choice(trades["계약일자"].unique(), 200, replace=False)
bad = 0
for d in pd.to_datetime(sample_dates):
    tr = trades[trades["계약일자"].eq(d)]
    known_year = d.year if d.month >= RELEASE_MONTH else d.year - 1
    g1 = gongsi_features(tr, price_lot[price_lot["기준연도"] <= known_year],
                         price_band[price_band["기준연도"] <= known_year])
    g2 = gongsi_features(tr, price_lot, price_band)
    z = zone_of.copy()
    for col, _ in STAGES:
        z.loc[z[col] >= d, col] = pd.NaT
    s1, s2 = stage_features(tr, z), stage_features(tr, zone_of)
    same = g1.equals(g2) and s1.equals(s2)
    bad += int(not same)
print("누수 의심 계약일:", bad)
assert bad == 0

edge = pd.DataFrame({"계약일자": pd.to_datetime(["2023-06-30", "2023-07-01"])})
assert list(available_year(edge["계약일자"])) == [2022, 2023]
print("✅ 공시가격·정비 단계: 계약일에 공개되지 않은 자료 미사용")

누수 의심 계약일: 0
✅ 공시가격·정비 단계: 계약일에 공개되지 않은 자료 미사용


## 5. 처음 보는 단지에 도움이 되는가 (Train 구간 점검)

공시가격 기반 추정가에 **직전 분기까지의 실거래/공시 비율**(서울 전체 중위)을 곱한 단순 추정이
실제 거래가를 얼마나 맞추는지 본다. 비교 기준은 V1 단순 기준선(같은 단지 최근 평당가 × 면적, MAPE 약 13%)과
V1 회귀의 처음 보는 단지 성능(MAPE 약 18%)이다. **처음 보는 단지**는 데이터에서 그 단지의 첫 거래로 근사한다.

In [9]:
# ============================================================
# 공시 기반 단순 추정의 정확도 (실제 거래금액은 비교용으로만 사용)
# ============================================================

price = pd.read_parquet(PROCESSED_DIR / "df_fe_base.parquet", columns=["transaction_id", "거래금액"])
ev = trades[["transaction_id", "단지일련번호", "계약일자"]].merge(gongsi_trade, on="transaction_id").merge(price, on="transaction_id")
ev["실제"] = ev["거래금액"].astype(float)
ev["분기"] = ev["계약일자"].dt.to_period("Q")
ev = ev[ev["계약일자"] < "2025-07-01"].dropna(subset=["공시기반_추정가"])

# 직전 분기까지 실거래/공시 비율 (서울 전체 중위) — 해당 분기 정보는 쓰지 않는다
ratio_q = (ev["실제"] / ev["공시기반_추정가"]).groupby(ev["분기"]).median()
ev["비율"] = ev["분기"].map(ratio_q.shift(1))
ev = ev.dropna(subset=["비율"])
ev["추정"] = ev["공시기반_추정가"] * ev["비율"]
ev["APE"] = (ev["추정"] / ev["실제"] - 1).abs() * 100

first = ev.sort_values("계약일자").drop_duplicates("단지일련번호")["transaction_id"]
ev["처음 보는 단지"] = ev["transaction_id"].isin(first)
display(ev.groupby("처음 보는 단지")["APE"].agg(["count", "mean", "median"]).round(2)
        .rename(columns={"mean": "MAPE(%)", "median": "Median APE(%)"}))

,count,MAPE(%),Median APE(%)
처음 보는 단지,,,
False,138134,10.02,7.17
True,6970,14.23,10.84


In [10]:
# ============================================================
# 단지 물리 속성: 같은 구·같은 분기 안에서 평당가와의 순위 상관 (참고)
# ============================================================

pp = pd.read_parquet(PROCESSED_DIR / "df_fe_base.parquet", columns=["transaction_id", "평당가"])
x = trades[["transaction_id", "단지일련번호", "시군구명", "계약일자"]].merge(pp, on="transaction_id")
x = x[x["계약일자"] < "2025-07-01"].merge(attrs[["단지일련번호"] + ATTR_FEATURES], on="단지일련번호", how="left")
x = x.merge(stage_trade[["transaction_id", "재건축_단계"]], on="transaction_id")
x["grp"] = x["시군구명"] + x["계약일자"].dt.to_period("Q").astype(str)
x["log평당가_구분기편차"] = np.log(x["평당가"].astype(float)) - x.groupby("grp")["평당가"].transform(lambda s: np.log(s.astype(float)).mean())

rows = []
for c in ATTR_FEATURES + ["재건축_단계"]:
    m = x[c].notna()
    rows.append({"feature": c, "Spearman": x.loc[m, c].rank().corr(x.loc[m, "log평당가_구분기편차"].rank()), "n": int(m.sum())})
display(pd.DataFrame(rows).sort_values("Spearman", key=abs, ascending=False).round(3))

,feature,Spearman,n
1,세대수,0.477,159908
0,동수,0.469,159871
2,최고층,0.397,159869
8,건폐율,-0.351,134728
4,지하층,-0.332,159871
5,승강기_100세대,-0.325,81466
3,평균층,0.308,159869
6,세대당_주차,0.261,136551
7,용적률,-0.098,134488
9,재건축_단계,0.085,160945


## 6. 저장

In [11]:
# ============================================================
# 저장
# - complex_attributes_v2: 단지 고정 속성
# - complex_trade_v2: 거래 단위 시점 Feature (공시가격·정비 단계). 통근시간은 모델 제외(웹 실시간 조회)
# - complex_official_price_v2: 단지 × 공시연도 (웹 표시·06 Merge)
# ============================================================

complex_attributes = attrs[["단지일련번호", "lot", "사용승인연도"] + ATTR_FEATURES].copy()
complex_trade = gongsi_trade.merge(stage_trade, on="transaction_id")
assert len(complex_trade) == len(trades) and complex_trade["transaction_id"].is_unique

official_by_complex = (complexes[["단지일련번호", "lot"]].merge(price_lot, on="lot", how="left")
                       .dropna(subset=["기준연도"]).astype({"기준연도": int}))

complex_attributes.to_parquet(PROCESSED_DIR / "complex_attributes_v2.parquet", index=False)
complex_trade.to_parquet(PROCESSED_DIR / "complex_trade_v2.parquet", index=False)
official_by_complex.to_parquet(PROCESSED_DIR / "complex_official_price_v2.parquet", index=False)

for name in ["complex_attributes_v2", "complex_trade_v2", "complex_official_price_v2"]:
    print(name, pd.read_parquet(PROCESSED_DIR / f"{name}.parquet").shape)

complex_attributes_v2 (7871, 12)
complex_trade_v2 (234302, 9)
complex_official_price_v2 (53196, 5)


## 정리

| 파일 | 단위 | 내용 |
|---|---|---|
| `complex_attributes_v2.parquet` | 단지 | 동 수·세대수·층수·지하층·승강기·세대당 주차·용적률·건폐율 |
| `complex_trade_v2.parquet` | 거래 | 공시가격(단지·면적대 ㎡당, 공시 기반 추정가, 전년 대비), 재건축 단계 |
| `complex_official_price_v2.parquet` | 단지 × 공시연도 | 단지 ㎡당 공시가격 중위·호 수 |

**한계**
- 건축물대장은 2026-10 snapshot이다. 물리 속성은 거의 변하지 않지만, 거래 시점 이후 리모델링이 있었다면 반영된다.
- 공시가격 포털 파일은 6월 조정공시 반영본이라 7월부터 사용한다(4월 최초 공시보다 2개월 늦게 반영).
- 정비사업 목록은 현재 진행 중인 구역만 담는다. 2021~2026년 사이 준공·해제된 구역의 과거 단계는 빠질 수 있다.
- **통근시간은 모델 Feature에서 제외한다 (2026-10-28 결정).**
  - ODsay 무료 할당량이 하루 약 30건으로 줄어 역 347곳 × 업무지구 3곳(1,041건) 수집에 한 달 이상 걸린다.
  - 공시가격이 이미 교통 프리미엄을 포함한 가격 수준을 담고 있고(처음 보는 단지 MAPE 18.4% → 14.2%),
    V1의 주요 일자리·역까지 직선거리 Feature도 있어 추가 개선 폭이 작을 것으로 본다.
  - ODsay는 현재 노선망만 반영해 개통 전 거래에는 시점 문제가 남는다.
  - 웹 단계에서 단지 상세 조회 시 단지 → 광화문·강남·여의도 통근시간을 ODsay로 실시간 조회·캐시해 보여준다.
    수집 스크립트와 받아 둔 30건(`data/external/commute/`)은 그때 재사용한다.